In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
torch.manual_seed(42)
batch_size=64
epochs=10

device=torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(f"Using device: {device}")


Using device: cpu


In [2]:

transform=transforms.ToTensor()
train_data=datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_data=datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
train_loader=DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)
test_loader=DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [3]:

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.Layers=nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )
    def forward(self,x):
        return self.Layers(x)
model=SimpleNN().to(device)

criterion=nn.CrossEntropyLoss()
optimiser=optim.Adam(model.parameters(),lr=0.001)


In [4]:

for epoch in range(epochs):
    model.train()
    running_loss=0.0
    
    correct = 0
    total = 0
    for images,labels in train_loader:
        images,labels=images.to(device),labels.to(device)
        
        logits=model(images)
        loss=criterion(logits,labels)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss+=loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch [1/10], Loss: 21.2986
Epoch 1/10 | Loss: 0.3330 | Accuracy: 90.62%
Epoch [2/10], Loss: 8.6662
Epoch 2/10 | Loss: 0.1355 | Accuracy: 95.97%
Epoch [3/10], Loss: 6.0459
Epoch 3/10 | Loss: 0.0945 | Accuracy: 97.19%
Epoch [4/10], Loss: 4.5849
Epoch 4/10 | Loss: 0.0717 | Accuracy: 97.78%
Epoch [5/10], Loss: 3.6116
Epoch 5/10 | Loss: 0.0565 | Accuracy: 98.21%
Epoch [6/10], Loss: 2.7637
Epoch 6/10 | Loss: 0.0432 | Accuracy: 98.65%
Epoch [7/10], Loss: 2.2531
Epoch 7/10 | Loss: 0.0352 | Accuracy: 98.83%
Epoch [8/10], Loss: 1.7875
Epoch 8/10 | Loss: 0.0279 | Accuracy: 99.13%
Epoch [9/10], Loss: 1.5446
Epoch 9/10 | Loss: 0.0241 | Accuracy: 99.20%
Epoch [10/10], Loss: 1.2360
Epoch 10/10 | Loss: 0.0193 | Accuracy: 99.36%


In [5]:

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Test accuracy: {100 * correct / total:.2f}%")

torch.save(model.state_dict(), "mnist_model.pth")
print("Model saved to mnist_model.pth")


Test accuracy: 97.88%
Model saved to mnist_model.pth


In [6]:

total_params = 0

for name, param in model.named_parameters():
    print(name, param.shape, param.numel())
    total_params += param.numel()

print("Total trainable parameters:", total_params)

Layers.1.weight torch.Size([128, 784]) 100352
Layers.1.bias torch.Size([128]) 128
Layers.3.weight torch.Size([64, 128]) 8192
Layers.3.bias torch.Size([64]) 64
Layers.5.weight torch.Size([10, 64]) 640
Layers.5.bias torch.Size([10]) 10
Total trainable parameters: 109386
